# Bone Fracture Detection: CNN vs ScatNet + Explainable AI

*Visual Intelligence 2025/2026, University of Verona*

Binary classification of bone X-rays (**fractured** vs **not fractured**) with a custom CNN, a
wavelet Scattering Network (ScatNet) and, as an extra baseline, a pretrained ResNet18. The
decisions are then explained with six XAI methods, and one of them (Occlusion) is also
implemented from scratch.

**All the code lives in `src/`.** This notebook only sets the parameters and calls those
functions, so it can be read top to bottom as the report of the experiment.

| Exam step | Section |
|---|---|
| dataset, train/test split | 1 |
| CNN and ScatNet (same classifier head) | 2 |
| k-fold CV: mean accuracy and mean F1 on the training set | 3 |
| test on the test set | 4 |
| extract and compare the filters | 5 |
| six XAI methods on both models, overlaid on 2 images per class | 6 |
| our implementation vs Captum | 7 |
| qualitative comparison of the methods | 8 |
| discussion | 9 |

In [ ]:
# Setup: works locally (notebook opened from notebooks/) and on Google Colab
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/bone-fracture-detection")  # copy of this repository
    %pip install -q kymatio captum
else:
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import data, models, training, xai, plots, utils

In [ ]:
# Settings: the only cell to edit
DATA_DIR    = PROJECT_ROOT / "Bone_Fracture_Binary_Classification" / "Bone_Fracture_Binary_Classification"
XAI_DIR     = DATA_DIR / "choosen_test"   # the 8 images we explain (4 per class)
RESULTS_DIR = PROJECT_ROOT / "results"    # weights, CV logs, attributions
FIG_DIR     = RESULTS_DIR / "figures"     # figures for the report

MODELS     = ["cnn", "resnet18", "scatnet"]
TRAIN      = False   # True: run the cross-validation and save the weights | False: load results/models/*.pth
K_FOLDS    = 5
EPOCHS     = 10
BATCH_SIZE = 32
LR         = 1e-3

XAI_PARAMS = dict(baseline=0.0,   # value given to "removed" pixels: 0.0 = grey, -1.0 = black
                  ig_steps=100, lime_samples=1000, shapley_samples=50,
                  occlusion_window=8, occlusion_stride=4)
XAI_SHOW   = [0, 1, 4, 5]         # 2 fractured + 2 not fractured images for the figures

utils.set_seed(42)
device = utils.get_device()
print("device:", device)

## 1. Data

Kaggle *Bone Fracture Multi-Region X-ray Data*, already split into train / val / test.
Cross-validation is run on `train` and the final evaluation on `test` (`val` is not used).
Images are resized to 224x224 and normalised to [-1, 1]; training images are also randomly
rotated (±15°), flipped and changed in brightness/contrast.

In [ ]:
data.summarize(DATA_DIR)

In [ ]:
datasets = data.load_datasets(DATA_DIR)
test_loader = data.make_loader(datasets["test"], BATCH_SIZE)
class_names = datasets["test"].class_names   # label 0 = fractured, 1 = not fractured

images, labels = next(iter(data.make_loader(datasets["train"], 8, shuffle=True)))
plots.show_images(images, labels, class_names, title="Training images (with augmentation)")

## 2. Models

| model | feature extractor | classifier |
|---|---|---|
| `cnn` | 4 learned conv blocks (32→64→128→256, BN, ReLU, max-pool) | FC 512 → 128 → 2 |
| `scatnet` | fixed Kymatio scattering transform (J=4 scales, L=8 orientations) | FC 512 (+BN) → 2 |
| `resnet18` | ImageNet-pretrained ResNet18 (extra baseline) | FC 512 → 2 |

ScatNet learns only its classifier, but that classifier sees 245,196 scattering coefficients,
which is why it has the most parameters.

> **Note:** the exam asks for the *same* classifier for CNN and ScatNet (only the number of input
> neurons may differ). The two heads currently differ (see the README, "Known issues").

In [ ]:
models.summary(MODELS)

## 3. 5-fold cross-validation (training set)

Stratified 5-fold CV. In each fold the model is trained for `EPOCHS` epochs (Adam, weight decay
1e-4, learning rate halved when the validation loss plateaus) and the epoch with the best
validation accuracy is kept. The weights of the best fold are saved and used from here on.
F1 is computed for the **fractured** class.

In [ ]:
cv = {}
for name in MODELS:
    if TRAIN:
        cv[name] = training.train_kfold_cv(name, datasets["train"], datasets["train_eval"], device,
                                           k_folds=K_FOLDS, epochs=EPOCHS, batch_size=BATCH_SIZE,
                                           lr=LR, results_dir=RESULTS_DIR)
    else:
        cv[name] = training.load_cv_results(name, RESULTS_DIR)
training.cv_table(cv)

In [ ]:
plots.plot_cv_history(cv, save_to=FIG_DIR / "cv_history.png")

## 4. Test set

In [ ]:
trained = {name: training.load_model(name, RESULTS_DIR, device) for name in MODELS}
test = {name: training.test_model(trained[name], test_loader, device) for name in MODELS}
training.results_table(cv, test)

In [ ]:
plots.plot_confusion_matrices(test, class_names, save_to=FIG_DIR / "confusion_matrices.png")

## 5. Filters: learned (CNN) vs fixed (ScatNet)

In [ ]:
plots.plot_conv_filters(trained["cnn"].conv1.weight, "Custom CNN: learned conv1 filters (3x3)",
                        save_to=FIG_DIR / "filters_cnn.png")

In [ ]:
plots.plot_scattering_filters(J=trained["scatnet"].J, L=trained["scatnet"].L,
                              save_to=FIG_DIR / "filters_scatnet.png")

## 6. Explainable AI

Six Captum methods, applied in the same way to every model:

| family | methods | what it measures |
|---|---|---|
| gradient | Saliency, Input x Gradient, Integrated Gradients | local sensitivity of the class score to each pixel |
| perturbation | LIME, Shapley (16x16 patches), Occlusion (8x8 window) | change of the class score when regions are removed |

Every map explains the **true class** of the image. In the overlays red = evidence for that
class, blue = evidence against it. The maps are cached in `results/attributions/` (delete the
file of a model to recompute it).

In [ ]:
xai_images, xai_labels = data.load_images(XAI_DIR)
xai_preds = {name: training.predict_labels(trained[name], xai_images, device) for name in MODELS}
training.prediction_table(xai_labels, xai_preds, class_names)

In [ ]:
attributions = {}
for name in MODELS:
    print(name)
    attributions[name] = xai.explain_all(trained[name], xai_images, xai_labels,
                                         cache=RESULTS_DIR / "attributions" / f"{name}.npz", **XAI_PARAMS)

In [ ]:
for name in MODELS:
    plots.plot_attribution_grid(xai_images, xai_labels, attributions[name], class_names,
                                methods=xai.CAPTUM_METHODS, preds=xai_preds[name], indices=XAI_SHOW,
                                title=f"{name}: six XAI methods", save_to=FIG_DIR / f"xai_{name}.png")

## 7. Our Occlusion vs Captum's Occlusion

`src/occlusion_scratch.py` re-implements Occlusion with plain PyTorch: slide an 8x8 window
(stride 4), replace it with the baseline, measure the drop of the class score, and average the
drops over all windows covering each pixel. With the same window grid and averaging as Captum
the two maps should be identical up to floating-point error.

In [ ]:
xai.scratch_vs_captum(attributions)

In [ ]:
plots.plot_scratch_vs_captum(xai_images, attributions, index=XAI_SHOW[0],
                             save_to=FIG_DIR / "occlusion_scratch_vs_captum.png")

## 8. Do the methods agree?

Spearman correlation between the maps of two methods (|attribution| averaged over 16x16
regions, mean over the 8 images). High = the two methods point at the same regions.

In [ ]:
agreement = {name: xai.agreement_matrix(attributions[name]) for name in MODELS}
plots.plot_agreement(agreement, save_to=FIG_DIR / "xai_agreement.png")

## 9. Discussion

*Numbers below are from the last full run (before the refactor); update them after re-running.*

| model | CV accuracy | CV F1 | test accuracy | test F1 (fractured) | test recall (fractured) |
|---|---|---|---|---|---|
| CNN | 97.4 ± 0.5 % | 97.4 % | 92.9 % | 92.4 % | 92.4 % |
| ResNet18 | 98.3 ± 0.7 % | 98.3 % | 94.7 % | 94.4 % | 95.0 % |
| ScatNet | 96.7 ± 0.5 % | 96.7 % | 90.5 % | 90.2 % | 92.4 % |

**Performance.** All models are well above the 75 % target. The pretrained ResNet18 is best,
the custom CNN is ~2 points behind and ScatNet another ~2 points behind, even though its
features are not learned at all. All models lose 3.7 to 6.2 points from CV to test: validation folds
are drawn from the same pool as the training images (which may contain augmented copies of the
same X-ray), so CV is optimistic and the test set is the reliable estimate.

**Filters.** The CNN's first-layer filters are tiny (3x3) and look noisy: they are generic
edge/contrast detectors shaped by the data. ScatNet uses Morlet wavelets fixed in advance, well
localised in scale (4 octaves) and orientation (8 angles): stable and interpretable, but not
adapted to fractures.

**XAI.** The three gradient methods agree with each other and give sharp, pixel-level maps along
bone edges; they are noisier for ScatNet, whose gradients pass through the fixed wavelets.
LIME and Shapley work on 16x16 patches: coarser but easier to read. With a grey baseline,
Occlusion lights up the black background around the bone for the CNNs: a grey square on a black
background is an unrealistic image, so the score changes for the wrong reason. A black baseline
(`XAI_PARAMS["baseline"] = -1.0`) removes this artefact.

**Methods that cannot be used on both models.** All six methods chosen here only need gradients
or forward passes, so they work on every model. Layer-based methods such as GradCAM need a
learned convolutional feature map (e.g. the CNN's `conv4`); ScatNet has none, only fixed wavelet
coefficients, so GradCAM would not show learned features there.

**Our implementation.** Our Occlusion reproduces Captum's (Pearson r = 1, differences at
floating-point level), which validates both the algorithm and our understanding of it.